# 01 Item 1A scraper

Downloads each firm's 10-K from SEC EDGAR and extracts Item 1A (Risk Factors). Output: `data/processed/item1a_10k_{year}.parquet`, failure logs in `data/logs/`. Input to Section 3.2 (sample construction).

In [1]:
# packages

from __future__ import annotations

import json
import re
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import warnings
import argparse

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

## Configuration and extraction functions

Extraction strategies, in order: EDGAR anchors, table-of-contents links, DOM heading scan, plain-text heading scan.

In [2]:
"""
Download S&P1500 10-K filings (2020–2025), extract Item 1A, and save one row per firm-year to Parquet.
"""

# -----------------------------
# CONFIG
# -----------------------------
USER_AGENT = "Soeren Schlisske soeren.schlisske@student.unisg.ch"
SP500_CIK_CSV = "SP1500_CIK.csv"
OUT_PARQUET = "data/processed/item1a_10k_2020_2025.parquet"
OUT_FAILURES = "data/logs/item1a_failures.csv"

REQUEST_SLEEP_S = 0.3
REQUEST_TIMEOUT_S = 60
MAX_RETRIES = 4
TRANSIENT = {429, 500, 502, 503, 504}

HEADERS_DATA = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov",
}
HEADERS_ARCHIVES = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}

# -----------------------------
# Helpers
# -----------------------------
def cik_10(cik: str | int) -> str:
    s = str(cik).strip().lstrip("0")
    return s.zfill(10)

def cik_archives(cik: str | int) -> str:
    return str(int(str(cik).strip()))

def safe_get(url, headers, timeout=60):
    last_status = None
    for attempt in range(1, MAX_RETRIES + 1):
        r = requests.get(url, headers=headers, timeout=timeout)
        last_status = r.status_code
        if r.status_code in TRANSIENT:
            retry_after = r.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                sleep_s = int(retry_after)
            else:
                sleep_s = min(60, (2 ** attempt) * REQUEST_SLEEP_S) + random.uniform(0, 0.5)
            time.sleep(sleep_s)
            continue
        r.raise_for_status()
        time.sleep(REQUEST_SLEEP_S)
        return r
    raise RuntimeError(f"GET failed after {MAX_RETRIES} retries: {url} (last status {last_status})")

def to_parquet_pyarrow(df: pd.DataFrame, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df2 = df.copy()
    for c in ["company_name", "filing_date", "report_date", "accession_nodash", "primary_doc", "source_url", "extract_version"]:
        if c in df2.columns:
            df2[c] = df2[c].astype("string")
    if "item_1a_text" in df2.columns:
        df2["item_1a_text"] = df2["item_1a_text"].astype("string")
    for c in ["cik", "fiscal_year", "item_1a_char_len"]:
        if c in df2.columns:
            df2[c] = pd.to_numeric(df2[c], errors="coerce").astype("Int64")
    table = pa.Table.from_pandas(df2, preserve_index=False)
    pq.write_table(table, path)

# -----------------------------
# SEC Submissions
# -----------------------------
def get_company_submissions(cik: str | int) -> Dict:
    url = f"https://data.sec.gov/submissions/CIK{cik_10(cik)}.json"
    return safe_get(url, HEADERS_DATA).json()

def get_historical_submission_file(name: str) -> Dict:
    url = f"https://data.sec.gov/submissions/{name}"
    return safe_get(url, HEADERS_DATA).json()

def filings_recent_to_df(submissions: Dict) -> pd.DataFrame:
    recent = submissions.get("filings", {}).get("recent", {})
    if not recent:
        return pd.DataFrame()
    return pd.DataFrame(recent)

def filings_all_to_df(submissions: Dict, start_date: str, end_date: str) -> pd.DataFrame:
    frames = []
    df_recent = filings_recent_to_df(submissions)
    if not df_recent.empty:
        frames.append(df_recent)
    for f in submissions.get("filings", {}).get("files", []) or []:
        name = f.get("name")
        if not name:
            continue
        if f.get("filingTo") and f["filingTo"] < start_date:
            continue
        if f.get("filingFrom") and f["filingFrom"] > end_date:
            continue
        hist = get_historical_submission_file(name)
        df_hist = filings_recent_to_df(hist)
        if not df_hist.empty:
            frames.append(df_hist)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates()

def list_10k_filings_for_year(submissions, target_year, filing_start, filing_end):
    df = filings_all_to_df(submissions, filing_start, filing_end)
    if df.empty:
        return df
    needed = ["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument"]
    for col in needed:
        if col not in df.columns:
            df[col] = None
    df = df[df["form"].eq("10-K")].copy()
    df = df[(df["filingDate"] >= filing_start) & (df["filingDate"] <= filing_end)].copy()

    def _year_from_reportdate(rd, fd):
        if isinstance(rd, str) and len(rd) >= 4:
            return int(rd[:4])
        if isinstance(fd, str) and len(fd) >= 7:
            fy, fm = int(fd[:4]), int(fd[5:7])
            return fy - 1 if 1 <= fm <= 4 else fy
        return None

    df["fiscal_year"] = [_year_from_reportdate(rd, fd) for rd, fd in zip(df["reportDate"], df["filingDate"])]
    df = df.dropna(subset=["fiscal_year"]).copy()
    df = df[df["fiscal_year"].astype(int).eq(int(target_year))].copy()
    if df.empty:
        return df
    df["accession_nodash"] = df["accessionNumber"].astype(str).str.replace("-", "", regex=False)
    df = df.sort_values(["fiscal_year", "filingDate"])
    df = df.groupby("fiscal_year", as_index=False).tail(1).reset_index(drop=True)
    return df[["form", "filingDate", "reportDate", "fiscal_year",
               "accessionNumber", "accession_nodash", "primaryDocument"]]

def download_filing_html(cik, accession_nodash, primary_doc) -> str:
    url = f"https://www.sec.gov/Archives/edgar/data/{cik_archives(cik)}/{accession_nodash}/{primary_doc}"
    return safe_get(url, HEADERS_ARCHIVES).text

# -----------------------------
# Item 1A extraction helpers
# -----------------------------
def make_soup_auto(raw: str) -> BeautifulSoup:
    head = raw.lstrip()[:2000].lower()
    is_true_xml = head.startswith("<?xml") or ("<xbrl" in head and "<html" not in head)
    return BeautifulSoup(raw, "lxml-xml" if is_true_xml else "lxml")

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def normalize_plain(s: str) -> str:
    return s.replace("\xa0", " ").replace("\u200b", " ").replace("\ufeff", "")

def looks_like_xbrl_instance(raw: str) -> bool:
    head = raw.lstrip()[:2000].lower()
    return ("<xbrl" in head) and ("<html" not in head)

def list_docs_from_index(cik, accession_nodash) -> list[dict]:
    cik_int = str(int(str(cik)))
    url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_nodash}/{accession_nodash}-index.html"
    soup = BeautifulSoup(safe_get(url, HEADERS_ARCHIVES).text, "lxml")
    best = None
    for t in soup.find_all("table"):
        header = t.get_text(" ", strip=True).lower()
        if "document" in header and "type" in header:
            best = t
            break
    if not best:
        return []
    docs = []
    for row in best.find_all("tr"):
        tds = row.find_all("td")
        if len(tds) < 4:
            continue
        doc = tds[2].get_text(" ", strip=True)
        typ = tds[3].get_text(" ", strip=True)
        size_txt = tds[4].get_text(" ", strip=True) if len(tds) >= 5 else ""
        if doc.lower().endswith((".htm", ".html", ".txt")):
            size_bytes = 0
            m = re.match(r"([\d\.]+)\s*(KB|MB|B)", size_txt.upper())
            if m:
                val, unit = float(m.group(1)), m.group(2)
                size_bytes = val * (1024 if unit == "KB" else 1024*1024 if unit == "MB" else 1)
            docs.append({"document": doc, "type": typ, "size_bytes": size_bytes})
    return docs

INDEX_CACHE = {}

def list_docs_from_index_cached(cik, accession_nodash):
    key = (int(cik), accession_nodash)
    if key not in INDEX_CACHE:
        INDEX_CACHE[key] = list_docs_from_index(cik, accession_nodash)
    return INDEX_CACHE[key]

def try_extract_item1a_from_accession(cik, accession_nodash, primary_doc):
    tried = []
    html = download_filing_html(cik, accession_nodash, primary_doc)
    tried.append((primary_doc, "PRIMARY"))
    item1a = extract_item_1a(html)
    if item1a:
        return item1a, primary_doc, tried
    try:
        docs = list_docs_from_index_cached(cik, accession_nodash)
    except Exception:
        return None, primary_doc, tried
    def sort_key(d):
        t = (d["type"] or "").upper()
        return (-1 if "10-K" in t else 0, -(d["size_bytes"] or 0))
    for d in sorted(docs, key=sort_key):
        doc = d["document"]
        if doc == primary_doc:
            continue
        html = download_filing_html(cik, accession_nodash, doc)
        tried.append((doc, d["type"]))
        item1a = extract_item_1a(html)
        if item1a:
            return item1a, doc, tried
    return None, primary_doc, tried

def anchor_target(href: str) -> Optional[str]:
    if href and href.startswith("#") and len(href) > 1:
        return href[1:]
    return None

def best_href_target_for_item(soup, item_re, extra_re=None) -> Optional[str]:
    """
    Score TOC href="#..." links. Penalises "Summary Risk Factors" / "Overview" entries
    heavily so the main Item 1A section anchor wins.
    """
    best = None
    best_score = -1
    for a in soup.find_all("a", href=True):
        target = anchor_target(a.get("href", ""))
        if not target:
            continue
        ctx = a.find_parent(["tr", "p", "div", "table"])
        ctx_txt  = clean_text(ctx.get_text(" ", strip=True)) if ctx else clean_text(a.get_text(" ", strip=True))
        link_txt = clean_text(a.get_text(" ", strip=True))

        score = 0
        if item_re.search(ctx_txt):  score += 3
        if item_re.search(link_txt): score += 2
        if extra_re:
            if extra_re.search(ctx_txt):  score += 2
            if extra_re.search(link_txt): score += 2

        # Heavy penalty for "Summary Risk Factors" / "Overview" TOC entries
        if re.search(r"\b(summar|overview)\b", link_txt, re.I): score -= 5
        if re.search(r"\b(summar|overview)\b", ctx_txt,  re.I): score -= 2

        if score > best_score:
            best_score = score
            best = target
    return best if best_score >= 2 else None

def iter_anchor_candidates(soup, target):
    cands = []
    cands.extend(soup.find_all(id=target))
    cands.extend(soup.find_all("a", attrs={"name": target}))
    cands.extend(soup.find_all(attrs={"name": target}))
    seen = set()
    return [c for c in cands if not (id(c) in seen or seen.add(id(c)))]

def peek_after(el, max_tags=40) -> str:
    texts, t = [], el
    for _ in range(max_tags):
        t = t.find_next()
        if not t:
            break
        if getattr(t, "name", None) in ("script", "style"):
            continue
        if hasattr(t, "get_text"):
            s = clean_text(t.get_text(" ", strip=True))
            if s:
                texts.append(s)
        if sum(len(x) for x in texts) > 6000:
            break
    return " ".join(texts)

def best_target_element(soup, target, want_pat):
    if not target:
        return None
    cands = iter_anchor_candidates(soup, target)
    if not cands:
        return None
    best, best_score = None, -1
    for c in cands:
        after = peek_after(c, max_tags=35)
        score = 0
        if want_pat.search(after[:2500]): score += 5
        if re.search(r"table of contents", after[:2500], re.I): score -= 3
        score += min(len(after) / 2000, 2)
        if score > best_score:
            best_score = score
            best = c
    return best if best_score >= 1 else cands[-1]

def collect_between_blocks(start_el, end_el, max_chars=350_000) -> str:
    parts, seen = [], set()
    for node in start_el.next_elements:
        if node is end_el:
            break
        if getattr(node, "name", None) in ("script", "style"):
            continue
        if not hasattr(node, "name"):
            continue
        if node.name in ("p", "div", "li", "tr", "td", "span", "font", "h1", "h2", "h3", "h4", "h5", "h6"):
            if id(node) in seen:
                continue
            seen.add(id(node))
            t = clean_text(node.get_text(" ", strip=True))
            if t:
                parts.append(t)
        if sum(len(p) for p in parts) > max_chars:
            break
    deduped = []
    for p in parts:
        if not deduped or p != deduped[-1]:
            deduped.append(p)
    return "\n".join(deduped)

def trim_to_item_1a_heading(text: str) -> str:
    text = normalize_plain(text)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    item1a_only      = re.compile(r"^item\s*1[\s\.]*\(?a[\.\-\u2013\u2014:\|]?\s*$", re.I)
    item1a_with_risk = re.compile(r"^item\s*1[\s\.]*\(?a[\.\-\u2013\u2014:\|\s]*risk\s*factors[\.\s]*$", re.I)
    risk_only        = re.compile(r"^risk\s*factors[\.\s]*$", re.I)
    toc_line         = re.compile(r"^table\s+of\s+contents\s*$", re.I)

    start_idx = None
    for i in range(len(lines)):
        if item1a_with_risk.match(lines[i]):
            start_idx = i
            break
        elif item1a_only.match(lines[i]):
            for offset in range(1, 6):
                if i + offset < len(lines) and risk_only.match(lines[i + offset]):
                    start_idx = i
                    break
            if start_idx is not None:
                break

    if start_idx is not None:
        return "\n".join(lines[start_idx:]).strip()

    for i, ln in enumerate(lines):
        if toc_line.match(ln):
            continue
        if risk_only.match(ln):
            return "\n".join(lines[i:]).strip()
        break

    item1a_embedded = re.compile(r"(?i)\bi\s*t\s*e\s*m\s*1\s*[\.\(]?\s*a\b")
    for i, ln in enumerate(lines[:50]):
        m = item1a_embedded.search(ln)
        if m:
            remainder = ln[m.start():].strip()
            rest = "\n".join(lines[i + 1:])
            return (remainder + ("\n" + rest if rest else "")).strip()

    return clean_text(text)

def cut_at_next_item(text: str) -> str:
    text = normalize_plain(text)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    end_heading = re.compile(r"^item\s*(1[\s\.]*[bc]|2)[\.\-\u2013\u2014:\|]?\s*$", re.I)
    for i in range(10, len(lines)):
        if end_heading.match(lines[i]):
            return clean_text("\n".join(lines[:i]))
    return clean_text("\n".join(lines))

def extract_by_line_headings(plain: str, min_len: int = 800) -> Optional[str]:
    plain = normalize_plain(plain)
    lines = [ln.strip() for ln in plain.splitlines()]

    item1a_only      = re.compile(r"^item\s*1[\s\.]*\(?a[\.\-\u2013\u2014:\|]?\s*$", re.I)
    item1a_with_risk = re.compile(r"^item\s*1[\s\.]*\(?a[\.\-\u2013\u2014:\|\s]*risk\s*factors[\.\s]*$", re.I)
    risk_only        = re.compile(r"^risk\s*factors[\.\s]*$", re.I)
    end_heading      = re.compile(r"^item\s*(1[\s\.]*[bc]|2)[\.\-\u2013\u2014:\|]?\s*$", re.I)
    item1a_emb_re    = re.compile(r"(?i)\bi\s*t\s*e\s*m\s*1\s*[\.\(]?\s*\(?a\b")

    candidates, fallback_cands = [], []
    for i in range(len(lines)):
        if item1a_with_risk.match(lines[i]):
            candidates.append(i)
        elif item1a_only.match(lines[i]):
            found_risk = False
            for offset in range(1, 6):
                if i + offset < len(lines) and risk_only.match(lines[i + offset]):
                    candidates.append(i)
                    found_risk = True
                    break
            if not found_risk:
                fallback_cands.append(i)

    seen_idxs = set(candidates + fallback_cands)
    emb_cands = [
        i for i, ln in enumerate(lines)
        if i not in seen_idxs and len(ln) < 150 and item1a_emb_re.search(ln)
    ]

    # Sliding-window multi-line scan — catches headings split across iXBRL sub-tags
    seen_idxs2 = set(candidates + fallback_cands + emb_cands)
    win_cands = []
    for i in range(len(lines)):
        if i in seen_idxs2:
            continue
        for width in range(2, 9):
            if i + width > len(lines):
                break
            chunk = [ln for ln in lines[i:i + width] if ln]
            if not chunk:
                break
            if any(len(ln) > 60 for ln in chunk):
                break
            joined = " ".join(chunk)
            if not (4 < len(joined) < 150):
                break
            if item1a_emb_re.search(joined):
                win_cands.append(i)
                seen_idxs2.add(i)
                break

    for start_i in candidates + fallback_cands + emb_cands + win_cands:
        end_i = None
        for j in range(start_i + 1, len(lines)):
            if end_heading.match(lines[j]):
                end_i = j
                break
        seg = "\n".join(lines[start_i:end_i]).strip() if end_i else "\n".join(lines[start_i:]).strip()
        seg = clean_text(seg)
        if len(seg) >= min_len:
            return seg

    return None

# -------------------------
# Shared S2/2b gate logic
# -------------------------
def _s2_accept(txt: str, min_len: int) -> bool:
    """
    Return True if txt looks like a valid Item 1A section.
    starts_ok — text begins with an iXBRL-tolerant heading/risk pattern
    has_risk  — "risk factor" phrase found within first 50 000 chars
    Large sections (> 50 K) that start correctly are accepted without phrase match,
    guarding against sections that describe risks without using the compound noun.
    """
    if len(txt) < min_len:
        return False
    starts_ok = bool(re.match(r"(?i)^(i\s*t\s*e\s*m\s*1[\s\.\(]*a|r\s*i\s*s\s*k|1[\s\.]*a\b)", txt))
    has_risk  = bool(re.search(r"r\s*i\s*s\s*k\s+f\s*a\s*c\s*t\s*o\s*r", txt[:50000], re.I))
    return (has_risk or (starts_ok and len(txt) > 50000)) and (starts_ok or len(txt) > 20000)

# -------------------------
# Main extractor
# -------------------------
def extract_item_1a(html: str, min_len: int = 800) -> Optional[str]:
    if looks_like_xbrl_instance(html):
        return None

    soup = make_soup_auto(html)

    # Strip ix:header only (NOT ix:hidden — needed by S2/2b for Donnelley-style filings)
    for tag in soup.find_all(["ix:header"]):
        tag.decompose()
    for tag in soup.find_all(lambda t: t.name in ("ix:header", "ixheader")):
        tag.decompose()

    # ── Strategy 1: EDGAR FIS anchors ────────────────────────────────────────
    fis = soup.find("a", attrs={"name": re.compile(r"FIS_RISK_FACTORS", re.I)})
    if fis:
        end = soup.find("a", attrs={"name": re.compile(
            r"FIS_UNRESOLVED_STAFF_COMMENTS|FIS_PROPERTIES|FIS_LEGAL_PROCEEDINGS|FIS_MINE_SAFETY_DISCLOSURES", re.I
        )})
        txt = collect_between_blocks(fis, end)
        txt = trim_to_item_1a_heading(txt)
        txt = cut_at_next_item(txt)
        txt = clean_text(txt)
        return txt if len(txt) >= min_len else None

    # ── Strategy 2: TOC href → anchor ────────────────────────────────────────
    item1a_re = re.compile(r"\bitem\s*1a\b", re.I)
    risk_re   = re.compile(r"risk\s*factors", re.I)

    start_target = (best_href_target_for_item(soup, item1a_re, risk_re)
                    or best_href_target_for_item(soup, item1a_re, None))
    end_target = (best_href_target_for_item(soup, re.compile(r"\bitem\s*1b\b", re.I), None)
                  or best_href_target_for_item(soup, re.compile(r"\bitem\s*1c\b", re.I), None)
                  or best_href_target_for_item(soup, re.compile(r"\bitem\s*2\b",  re.I), None))

    start_el = best_target_element(soup, start_target, re.compile(r"\bitem\s*1a\b", re.I))
    end_el   = best_target_element(soup, end_target,   re.compile(r"\bitem\s*(1b|1c|2)\b", re.I)) if end_target else None

    if start_el:
        txt = collect_between_blocks(start_el, end_el)
        txt = trim_to_item_1a_heading(txt)
        txt = cut_at_next_item(txt)
        txt = clean_text(txt)
        if _s2_accept(txt, min_len):
            return txt

    # ── Strategy 2b: DOM element scan (no TOC anchor needed) ─────────────────
    # For filings like Intel / WFC that have no href-based TOC links.
    # Scans all short text elements for "Item 1A" and picks the one with the most
    # content after it. ix:hidden is still intact so collect_between_blocks sees
    # everything (including content stored as XBRL-tagged hidden facts).
    if not start_el:
        _i1a_scan = re.compile(r"(?i)\bi\s*t\s*e\s*m\s*1\s*[\.\(]?\s*\(?a\b")
        _dom_candidates = []
        for tag in soup.find_all(["p", "div", "h1", "h2", "h3", "h4", "h5", "span", "td", "font"]):
            t_txt = clean_text(tag.get_text(" ", strip=True))
            if 2 < len(t_txt) < 200 and _i1a_scan.search(t_txt):
                _dom_candidates.append(tag)

        _best_el, _best_score = None, -1
        for el in _dom_candidates:
            after = peek_after(el, max_tags=80)
            score = min(len(after) / 500, 10)
            if re.search(r"table of contents", after[:2000], re.I):
                score -= 5
            if score > _best_score:
                _best_score, _best_el = score, el

        if _best_el and _best_score > 0:
            txt = collect_between_blocks(_best_el, None)
            txt = trim_to_item_1a_heading(txt)
            txt = cut_at_next_item(txt)
            txt = clean_text(txt)
            if _s2_accept(txt, min_len):
                return txt

    # ── Strategy 3: plain-text line-heading scan ──────────────────────────────
    # Strip ix:hidden here — safe because S2 / S2b already ran.
    # ix:hidden can produce duplicate headings in get_text() that make end_heading
    # fire prematurely, yielding a too-short segment and a false miss.
    for tag in soup.find_all(lambda t: getattr(t, "name", "") in ("ix:hidden", "ixhidden")):
        tag.decompose()

    plain = soup.get_text("\n", strip=True)
    return extract_by_line_headings(plain, min_len=min_len)

# -----------------------------
# Index / filing helpers
# -----------------------------
def detect_columns(df: pd.DataFrame) -> Tuple[str, str]:
    cik_candidates  = [c for c in df.columns if c.lower() in ["cik", "cik_str", "ciknumber"]]
    name_candidates = [c for c in df.columns if c.lower() in ["company_name", "name", "title", "security", "company"]]
    if not cik_candidates:
        raise ValueError(f"Could not find CIK column. Columns: {list(df.columns)}")
    return cik_candidates[0], name_candidates[0] if name_candidates else None

# -----------------------------
# Main loop
# -----------------------------
def run(target_year: int, max_firms: int | None = None):
    out_parquet  = f"data/processed/item1a_10k_{target_year}.parquet"
    out_failures = f"data/logs/item1a_failures_{target_year}.csv"
    filing_start = f"{target_year}-01-01"
    filing_end   = f"{target_year + 1}-12-31"

    Path(out_parquet).parent.mkdir(parents=True, exist_ok=True)
    Path(out_failures).parent.mkdir(parents=True, exist_ok=True)

    sp = pd.read_csv(SP500_CIK_CSV)
    cik_col, name_col = detect_columns(sp)
    rows, failures = [], []
    n = len(sp) if max_firms is None else min(len(sp), max_firms)

    for idx, r in sp.iloc[:n].iterrows():
        cik = r[cik_col]
        company_name = r[name_col] if name_col else None

        try:
            submissions = get_company_submissions(cik)
            filings = list_10k_filings_for_year(submissions, target_year, filing_start, filing_end)

            if filings.empty:
                failures.append({
                    "cik": cik, "company_name": company_name,
                    "reason": "no_10k_for_target_fy",
                    "detail": f"fy={target_year} window={filing_start}..{filing_end}"
                })
                continue

            for _, f in filings.iterrows():
                accession_nodash = f["accession_nodash"]
                primary_doc      = f["primaryDocument"]
                filing_date      = f["filingDate"]
                report_date      = f["reportDate"]
                fiscal_year      = int(f["fiscal_year"])

                try:
                    item1a, used_doc, tried_docs = try_extract_item1a_from_accession(
                        cik, accession_nodash, primary_doc
                    )
                    if not item1a:
                        failures.append({
                            "cik": cik, "company_name": company_name,
                            "reason": "item1a_not_found",
                            "detail": (f"{fiscal_year} {filing_date} {accession_nodash} primary={primary_doc} "
                                       f"tried={len(tried_docs)} last_used={used_doc} docs={[d[0] for d in tried_docs[:5]]}")
                        })
                        continue

                    rows.append({
                        "cik": int(str(cik).strip()),
                        "company_name": company_name,
                        "fiscal_year": fiscal_year,
                        "filing_date": filing_date,
                        "report_date": report_date,
                        "accession_nodash": accession_nodash,
                        "primary_doc": primary_doc,
                        "source_url": (f"https://www.sec.gov/Archives/edgar/data/"
                                       f"{cik_archives(cik)}/{accession_nodash}/{primary_doc}"),
                        "item_1a_text": item1a,
                        "item_1a_char_len": len(item1a),
                        "extract_version": "v5_fixed",
                    })

                except Exception as e:
                    failures.append({
                        "cik": cik, "company_name": company_name,
                        "reason": "download_or_parse_error",
                        "detail": f"{fiscal_year} {filing_date} {accession_nodash} {primary_doc} | {repr(e)}"
                    })

        except Exception as e:
            failures.append({
                "cik": cik, "company_name": company_name,
                "reason": "submissions_error", "detail": repr(e)
            })

        if (idx + 1) % 25 == 0:
            print(f"[progress] {idx+1}/{n} firms | rows={len(rows)} | failures={len(failures)}")

    df_out = pd.DataFrame(rows)
    to_parquet_pyarrow(df_out, out_parquet)
    pd.DataFrame(failures).to_csv(out_failures, index=False)
    print(f"[done] firms={n} | rows={len(df_out)} saved to {out_parquet} | failures={len(failures)} saved to {out_failures}")

In [3]:
CHECKPOINT_EVERY = 25  # flush to disk every N firms


def run(target_year: int, max_firms: int | None = None):
    """
    Scrape Item 1A for all S&P 1500 firms for target_year.

    Checkpointing: progress is flushed to disk every CHECKPOINT_EVERY firms.
    If interrupted, re-running will skip already-completed firms and continue
    from where it left off. Checkpoint files are deleted on successful completion.

    Checkpoint files:
      data/intermediate/item1a_10k_{year}_ckpt.parquet
      data/intermediate/item1a_failures_{year}_ckpt.csv
    """
    out_parquet  = f"data/processed/item1a_10k_{target_year}.parquet"
    out_failures = f"data/logs/item1a_failures_{target_year}.csv"
    ckpt_rows    = f"data/intermediate/item1a_10k_{target_year}_ckpt.parquet"
    ckpt_fails   = f"data/intermediate/item1a_failures_{target_year}_ckpt.csv"
    filing_start = f"{target_year}-01-01"
    filing_end   = f"{target_year + 1}-12-31"

    for p in [out_parquet, out_failures, ckpt_rows]:
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    sp = pd.read_csv(SP500_CIK_CSV)
    cik_col, name_col = detect_columns(sp)
    n = len(sp) if max_firms is None else min(len(sp), max_firms)
    firms = sp.iloc[:n]

    # ── Load checkpoint state ────────────────────────────────────────────────
    done_ciks = set()

    if Path(ckpt_rows).exists():
        df_ckpt = pd.read_parquet(ckpt_rows)
        done_ciks.update(int(c) for c in df_ckpt["cik"].dropna())
        print(f"[resume] checkpoint: {len(df_ckpt)} rows ({len(done_ciks)} firms already scraped)")
    else:
        df_ckpt = pd.DataFrame()

    if Path(ckpt_fails).exists():
        df_fail_ckpt = pd.read_csv(ckpt_fails)
        done_ciks.update(int(c) for c in df_fail_ckpt["cik"].dropna())
        print(f"[resume] checkpoint: {len(df_fail_ckpt)} failures already recorded")
    else:
        df_fail_ckpt = pd.DataFrame()

    remaining = firms[~firms[cik_col].astype(int).isin(done_ciks)]
    skipped = len(firms) - len(remaining)
    if skipped:
        print(f"[resume] skipping {skipped} already-done firms — {len(remaining)} remaining")

    # ── Flush helper ─────────────────────────────────────────────────────────
    new_rows, new_failures = [], []

    def flush():
        nonlocal df_ckpt, df_fail_ckpt
        if new_rows:
            df_ckpt = pd.concat([df_ckpt, pd.DataFrame(new_rows)], ignore_index=True)
            to_parquet_pyarrow(df_ckpt, ckpt_rows)
            new_rows.clear()
        if new_failures:
            df_fail_ckpt = pd.concat([df_fail_ckpt, pd.DataFrame(new_failures)], ignore_index=True)
            df_fail_ckpt.to_csv(ckpt_fails, index=False)
            new_failures.clear()

    # ── Main loop ────────────────────────────────────────────────────────────
    for i, (_, r) in enumerate(remaining.iterrows()):
        cik = r[cik_col]
        company_name = r[name_col] if name_col else None

        try:
            submissions = get_company_submissions(cik)
            filings = list_10k_filings_for_year(submissions, target_year, filing_start, filing_end)

            if filings.empty:
                new_failures.append({
                    "cik": cik, "company_name": company_name,
                    "reason": "no_10k_for_target_fy",
                    "detail": f"fy={target_year} window={filing_start}..{filing_end}",
                })
            else:
                for _, f in filings.iterrows():
                    accession_nodash = f["accession_nodash"]
                    primary_doc      = f["primaryDocument"]
                    filing_date      = f["filingDate"]
                    report_date      = f["reportDate"]
                    fiscal_year      = int(f["fiscal_year"])

                    try:
                        item1a, used_doc, tried_docs = try_extract_item1a_from_accession(
                            cik, accession_nodash, primary_doc
                        )
                        if not item1a:
                            new_failures.append({
                                "cik": cik, "company_name": company_name,
                                "reason": "item1a_not_found",
                                "detail": (f"{fiscal_year} {filing_date} {accession_nodash} primary={primary_doc} "
                                           f"tried={len(tried_docs)} last_used={used_doc} docs={[d[0] for d in tried_docs[:5]]}"),
                            })
                        else:
                            new_rows.append({
                                "cik":              int(str(cik).strip()),
                                "company_name":     company_name,
                                "fiscal_year":      fiscal_year,
                                "filing_date":      filing_date,
                                "report_date":      report_date,
                                "accession_nodash": accession_nodash,
                                "primary_doc":      primary_doc,
                                "source_url":       (f"https://www.sec.gov/Archives/edgar/data/"
                                                     f"{cik_archives(cik)}/{accession_nodash}/{primary_doc}"),
                                "item_1a_text":     item1a,
                                "item_1a_char_len": len(item1a),
                                "extract_version":  "v5_fixed",
                            })
                    except Exception as e:
                        new_failures.append({
                            "cik": cik, "company_name": company_name,
                            "reason": "download_or_parse_error",
                            "detail": f"{fiscal_year} {filing_date} {accession_nodash} {primary_doc} | {repr(e)}",
                        })

        except Exception as e:
            new_failures.append({
                "cik": cik, "company_name": company_name,
                "reason": "submissions_error", "detail": repr(e),
            })

        if (i + 1) % CHECKPOINT_EVERY == 0:
            flush()
            total_done = skipped + i + 1
            print(f"[checkpoint] {total_done}/{n} firms | rows={len(df_ckpt)} | failures={len(df_fail_ckpt)}")

    flush()  # write any remaining buffer

    # ── Finalize: write permanent outputs, delete checkpoints ────────────────
    to_parquet_pyarrow(df_ckpt, out_parquet)
    df_fail_ckpt.to_csv(out_failures, index=False)
    Path(ckpt_rows).unlink(missing_ok=True)
    Path(ckpt_fails).unlink(missing_ok=True)
    print(f"[done] year={target_year} | rows={len(df_ckpt)} → {out_parquet} | failures={len(df_fail_ckpt)} → {out_failures}")


## Retry failed extractions

In [4]:
def run_on_failures(target_year: int):
    """
    Retry all item1a_not_found failures for target_year using the updated extraction logic.

    - Reads  data/logs/item1a_failures_{target_year}.csv
    - Retries every row with reason == 'item1a_not_found'
    - Merges recovered rows INTO the existing main parquet (no separate file)
    - Rewrites the failures CSV: item1a_not_found rows are replaced by whatever
      still failed; no_10k_for_target_fy rows are preserved unchanged
    """
    main_parquet  = f"data/processed/item1a_10k_{target_year}.parquet"
    failures_csv  = f"data/logs/item1a_failures_{target_year}.csv"

    fails = pd.read_csv(failures_csv)
    todo  = fails[fails["reason"] == "item1a_not_found"].copy()
    keep  = fails[fails["reason"] != "item1a_not_found"].copy()   # preserve other reasons

    print(f"[retry] {len(todo)} item1a_not_found to retry  ({len(keep)} other failures preserved)")

    # detail format written by run():
    #   "{fiscal_year} {filing_date} {accession_nodash} primary={primary_doc} tried=..."
    pat = re.compile(
        r"(?P<fiscal_year>\d{4})\s+"
        r"(?P<filing_date>\d{4}-\d{2}-\d{2})\s+"
        r"(?P<accession_nodash>\d{18})\s+"
        r"primary=(?P<primary_doc>\S+?)\s"   # strip the "primary=" prefix
    )

    recovered    = []
    still_failed = []

    for i, (_, row) in enumerate(todo.iterrows(), 1):
        cik          = row["cik"]
        company_name = row.get("company_name")
        detail       = str(row.get("detail", ""))

        m = pat.search(detail)
        if not m:
            still_failed.append(row.to_dict())
            print(f"  [!] could not parse detail for {company_name}: {detail[:80]}")
            continue

        fiscal_year      = int(m.group("fiscal_year"))
        filing_date      = m.group("filing_date")
        accession_nodash = m.group("accession_nodash")
        primary_doc      = m.group("primary_doc")

        try:
            item1a, used_doc, tried_docs = try_extract_item1a_from_accession(
                cik, accession_nodash, primary_doc
            )

            if item1a:
                recovered.append({
                    "cik":              int(str(cik).strip()),
                    "company_name":     company_name,
                    "fiscal_year":      fiscal_year,
                    "filing_date":      filing_date,
                    "report_date":      None,
                    "accession_nodash": accession_nodash,
                    "primary_doc":      primary_doc,
                    "source_url":       f"https://www.sec.gov/Archives/edgar/data/"
                                        f"{cik_archives(cik)}/{accession_nodash}/{primary_doc}",
                    "item_1a_text":     item1a,
                    "item_1a_char_len": len(item1a),
                    "extract_version":  "v2_fixed",
                })
            else:
                still_failed.append({
                    **row.to_dict(),
                    "reason": "item1a_not_found",
                    "detail": (f"{fiscal_year} {filing_date} {accession_nodash} "
                               f"primary={primary_doc} tried={len(tried_docs)} last_used={used_doc}"),
                })

        except Exception as e:
            still_failed.append({
                **row.to_dict(),
                "reason": "retry_error",
                "detail": f"{fiscal_year} {filing_date} {accession_nodash} {primary_doc} | {repr(e)}",
            })

        if i % 10 == 0:
            print(f"  [{i}/{len(todo)}] recovered={len(recovered)}  still_failed={len(still_failed)}")

    print(f"\n[retry] done — recovered={len(recovered)}  still_failed={len(still_failed)}")

    # ── merge recovered rows into the existing main parquet ──────────────────
    if recovered:
        df_new = pd.DataFrame(recovered)
        if Path(main_parquet).exists():
            df_existing = pd.read_parquet(main_parquet)
            df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        else:
            df_combined = df_new
        to_parquet_pyarrow(df_combined, main_parquet)
        print(f"[retry] main parquet → {len(df_combined)} total rows  ({main_parquet})")

    # ── rewrite failures CSV: keep other reasons + new still_failed ──────────
    df_updated = pd.concat(
        [keep, pd.DataFrame(still_failed)], ignore_index=True
    )
    df_updated.to_csv(failures_csv, index=False)
    print(f"[retry] failures CSV → {len(df_updated)} remaining  ({failures_csv})")

    return pd.DataFrame(recovered), pd.DataFrame(still_failed)

## Run the scraper

In [5]:
YEARS     = [2018]   # 2014 through 2025 inclusive
MAX_FIRMS = None                # set to an int to limit firms per year (testing)

for year in YEARS:
    out_parquet = f"data/processed/item1a_10k_{year}.parquet"
    ckpt_rows   = f"data/intermediate/item1a_10k_{year}_ckpt.parquet"

    if Path(out_parquet).exists():
        print(f"[skip] {year} already done — {out_parquet}")
        continue

    if Path(ckpt_rows).exists():
        print(f"\n{'='*60}\n[resume] year={year} — checkpoint found, continuing\n{'='*60}")
    else:
        print(f"\n{'='*60}\n[run] year={year}\n{'='*60}")

    run(target_year=year, max_firms=MAX_FIRMS)
    run_on_failures(year)

    INDEX_CACHE.clear()

print("\n[all done]")



[run] year=2018
[checkpoint] 25/1498 firms | rows=19 | failures=6
[checkpoint] 50/1498 firms | rows=35 | failures=15
[checkpoint] 75/1498 firms | rows=51 | failures=24
[checkpoint] 100/1498 firms | rows=72 | failures=28
[checkpoint] 125/1498 firms | rows=88 | failures=37
[checkpoint] 150/1498 firms | rows=107 | failures=43
[checkpoint] 175/1498 firms | rows=123 | failures=52
[checkpoint] 200/1498 firms | rows=141 | failures=59
[checkpoint] 225/1498 firms | rows=158 | failures=67
[checkpoint] 250/1498 firms | rows=176 | failures=74
[checkpoint] 275/1498 firms | rows=196 | failures=79
[checkpoint] 300/1498 firms | rows=218 | failures=82
[checkpoint] 325/1498 firms | rows=234 | failures=91
[checkpoint] 350/1498 firms | rows=251 | failures=99
[checkpoint] 375/1498 firms | rows=267 | failures=108
[checkpoint] 400/1498 firms | rows=285 | failures=115
[checkpoint] 425/1498 firms | rows=301 | failures=124
[checkpoint] 450/1498 firms | rows=320 | failures=130
[checkpoint] 475/1498 firms | rows

## Diagnostics of failed extractions

In [ ]:
# ============================================================
# DIAGNOSTIC CELL — run this to trace extraction failures
# Reads the failures CSV, downloads the first N item1a_not_found
# filings, and prints a step-by-step trace for each one.
# ============================================================
import re, textwrap[]
import pandas as pd

DIAG_N = 10   # how many failing filings to inspect

fails = pd.read_csv("data/logs/item1a_failures_2024.csv")
nf = fails[fails["reason"] == "item1a_not_found"].head(15)

# detail format: "{fiscal_year} {filing_date} {accession_nodash} primary={primary_doc} tried=..."
pat = re.compile(r"(\d{4})\s+(\d{4}-\d{2}-\d{2})\s+(\d{18})\s+primary=(\S+?)\s")

for _, row in nf.iterrows():
    cik   = row["cik"]
    name  = row["company_name"]
    m     = pat.search(str(row["detail"]))
    if not m:
        print(f"  [skip] couldn't parse detail for {name}: {row['detail'][:100]}")
        continue

    acc, doc = m.group(3), m.group(4)
    url = f"https://www.sec.gov/Archives/edgar/data/{cik_archives(cik)}/{acc}/{doc}"

    print(f"\n{'='*70}")
    print(f"  {name}  |  CIK {cik}  |  {doc}")
    print(f"  {url}")

    try:
        html = download_filing_html(cik, acc, doc)
    except Exception as e:
        print(f"  [DOWNLOAD ERROR] {e}")
        continue

    soup = make_soup_auto(html)
    for tag in soup.find_all(["ix:header"]):
        tag.decompose()
    for tag in soup.find_all(lambda t: t.name in ("ix:header", "ixheader")):
        tag.decompose()

    # --- metadata ---
    is_ixbrl   = "ix:" in html[:8000].lower() or "xmlns:ix" in html[:3000].lower()
    ix_hidden  = len(re.findall(r"<ix:hidden", html, re.I))
    ix_header  = len(re.findall(r"<ix:header", html, re.I))
    has_fis    = bool(re.search(r"FIS_RISK_FACTORS", html, re.I))
    print(f"  iXBRL={is_ixbrl}  ix:header={ix_header}  ix:hidden={ix_hidden}  FIS={has_fis}")

    # --- strategy 1: FIS anchors ---
    fis = soup.find("a", attrs={"name": re.compile(r"FIS_RISK_FACTORS", re.I)})
    print(f"  [S1] FIS anchor found: {fis is not None}")

    # --- strategy 2: TOC href scan ---
    item1a_re = re.compile(r"\bitem\s*1a\b", re.I)
    risk_re   = re.compile(r"risk\s*factors", re.I)
    start_target = (best_href_target_for_item(soup, item1a_re, risk_re)
                    or best_href_target_for_item(soup, item1a_re, None))
    end_target = (best_href_target_for_item(soup, re.compile(r"\bitem\s*1b\b", re.I), None)
                  or best_href_target_for_item(soup, re.compile(r"\bitem\s*2\b", re.I), None))
    print(f"  [S2] TOC start_target={start_target!r}  end_target={end_target!r}")

    if start_target:
        start_el = best_target_element(soup, start_target, re.compile(r"\bitem\s*1a\b", re.I))
        end_el   = best_target_element(soup, end_target,   re.compile(r"\bitem\s*(1b|2)\b", re.I)) if end_target else None
        print(f"  [S2] start_el tag={getattr(start_el,'name',None)}  end_el tag={getattr(end_el,'name',None)}")

        if start_el:
            raw = collect_between_blocks(start_el, end_el)
            trimmed = trim_to_item_1a_heading(raw)
            cut = cut_at_next_item(trimmed)
            final = clean_text(cut)
            starts_ok = re.match(r"(?i)^(item\s*1a|risk\s*factors)", final)
            has_risk  = re.search(r"risk\s*factors", final[:10000], re.I)
            print(f"  [S2] collect len={len(raw)}  trimmed len={len(trimmed)}  final len={len(final)}")
            print(f"  [S2] starts_ok={bool(starts_ok)}  has_risk={bool(has_risk)}")
            print(f"  [S2] first 200 chars of final: {repr(final[:200])}")

    # --- strategy 3: line-heading scan ---
    plain = soup.get_text("\n", strip=True)
    item1a_re2 = re.compile(r"^item\s*1a[\.\-—:\|]?\s*$", re.I)
    item1awrisk = re.compile(r"^item\s*1a[\.\-—:\|\s]*risk\s*factors\s*$", re.I)
    lines = [l.strip() for l in plain.splitlines()]
    cands = [(i, ln) for i, ln in enumerate(lines) if item1awrisk.match(ln) or item1a_re2.match(ln)]
    print(f"  [S3] item1a heading candidates in plain text: {cands[:6]}")
    txt3 = extract_by_line_headings(plain, min_len=800)
    print(f"  [S3] extract_by_line_headings result len: {len(txt3) if txt3 else 0}")

    # --- raw TOC anchor links mentioning item 1a or risk factors ---
    toc_links = []
    for a in soup.find_all("a", href=True):
        t = anchor_target(a.get("href",""))
        if not t: continue
        txt_a = clean_text(a.get_text(" ", strip=True))
        if re.search(r"item\s*1a|risk\s*factor", txt_a, re.I):
            toc_links.append((t, txt_a[:60]))
    print(f"  [TOC] item1a anchor links: {toc_links[:6]}")


  Intel Corp  |  CIK 50863  |  intc-20241228.htm
  https://www.sec.gov/Archives/edgar/data/50863/000005086325000009/intc-20241228.htm
  iXBRL=True  ix:header=1  ix:hidden=1  FIS=False
  [S1] FIS anchor found: False
  [S2] TOC start_target=None  end_target=None
  [S3] item1a heading candidates in plain text: [(8248, 'Item 1A.')]
  [S3] extract_by_line_headings result len: 0
  [TOC] item1a anchor links: []

  Wells Fargo & Company  |  CIK 72971  |  wfc-20241231.htm
  https://www.sec.gov/Archives/edgar/data/72971/000007297125000066/wfc-20241231.htm
  iXBRL=True  ix:header=1  ix:hidden=1  FIS=False
  [S1] FIS anchor found: False
  [S2] TOC start_target=None  end_target=None
  [S3] item1a heading candidates in plain text: [(256, 'ITEM 1A.')]
  [S3] extract_by_line_headings result len: 0
  [TOC] item1a anchor links: []
